# Ye 2026 sepsis claims — panel-matched re-scoring (2026-10-12)

Re-run of the fixed-marker claim test in `paper_claim_analysis/run_fixed_claims_k_comparison.py` with one change requested:
**every author marker list is restricted to the genes measured in the PBISC 6k panel, and the same restricted list scores real and
generated cells** (both normalised as log1p CP10k over the 5,853 panel genes). Thresholds were refitted with the identical
procedure (5-fold sample-held-out CV, sample-balanced Youden, seed 20261012). Generated cells are compared with real cells scored by
**the same classifier** (classifier-positive fraction within parent), and with the real exact-label fraction for reference.

Inputs: recapitulated Ye object (in_paper_scheme cells, 133 samples), all five draws of K = 1,000 (183,000 cells each) and K = 4,000
(488,000 cells each). Parents: real = recap label prefix; generated = AIFI L2 mapping (SEP-3 has only a DN-T proxy). Sensitivity
variant B fits thresholds on real cells binomially thinned to generated panel depth. Code: `code/r_rescore.py`, `code/s_summary.py`.

In [ ]:
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
R = 'results/rescore/'
display(pd.read_csv(R + 'panel_marker_coverage.csv')[['claim', 'author_markers', 'panel_markers']])

## 1. Refitted thresholds: sample-held-out performance (full-panel AUC from the 09:15 run for comparison)

In [ ]:
cv = pd.read_csv(R + 'thresholds_cv_panel_matched.csv')
cv[['claim_id','variant','threshold','cv_auc','cv_sensitivity','cv_specificity','previous_full_panel_auc','panel_markers','author_markers']].round(3)

## 2. Adult healthy control vs adult sepsis: real (label and same classifier) vs generated

In [ ]:
S = pd.read_csv(R + 'summary_panel_matched.csv')
A = S[S.variant == 'A_panel'].set_index('claim_id')
cols = ['cv_auc','real_label_HC_median','real_label_sepsis_median','real_classifier_HC_median','real_classifier_sepsis_median',
        'real_classifier_diff','real_classifier_lo','real_classifier_hi','k1k_classifier_HC_median','k1k_classifier_sepsis_median',
        'k1k_classifier_diff','k1k_classifier_lo','k1k_classifier_hi','k4k_classifier_HC_median','k4k_classifier_sepsis_median',
        'k4k_classifier_diff','k4k_classifier_lo','k4k_classifier_hi','K4_minus_K1_diff','K4_minus_K1_lo','K4_minus_K1_hi']
A[cols].round(3).T

### Threshold-free mean marker score in the parent lineage, and per-draw contrasts

In [ ]:
A[['real_score_diff','real_score_lo','real_score_hi','k1k_score_diff','k1k_score_lo','k1k_score_hi','k4k_score_diff','k4k_score_lo','k4k_score_hi',
   'real_score_HC_median','k1k_score_HC_median','k4k_score_HC_median','k1k_per_draw_diff','k4k_per_draw_diff']].round(3)

In [ ]:
display(Image('figs/rescore_contrasts.png'))

## 3. Sensitivity: thresholds fitted on depth-matched real cells (variant B)

In [ ]:
B = S[S.variant == 'B_panel_depth'].set_index('claim_id')
B[['cv_auc','real_classifier_HC_median','real_classifier_sepsis_median','real_classifier_diff','k1k_classifier_HC_median','k1k_classifier_sepsis_median',
   'k1k_classifier_diff','k4k_classifier_HC_median','k4k_classifier_sepsis_median','k4k_classifier_diff']].round(3)

## 4. Recompute a contrast from the per-sample tables (check)

In [ ]:
real = pd.read_csv(R + 'real_by_sample.csv'); gen = pd.read_csv(R + 'generated_by_sample_draw.csv')
c = 'SEP-1'
r = real[(real.claim_id == c) & (real.n_parent >= 20)]
g = gen[(gen.claim_id == c) & (gen.n_parent >= 20)].groupby(['K','sample','group']).posA_frac.mean().reset_index()
print('real classifier', r.groupby('group').posA_frac.median().round(3).to_dict())
print('real label     ', r.groupby('group').label_frac.median().round(3).to_dict())
print(g.groupby(['K','group']).posA_frac.median().round(3))

## Reading

* Panel restriction leaves the classifiers about as weak as before (AUC 0.68–0.83 for SEP-1, SEP-2, SEP-3, SEP-3-NK, SEP-4B); SEP-4A (0.49) and SEP-5 (0.40)
  still do not separate their target cells, so they cannot be tested this way.
* Scored by the same classifier, real cells show the classifier-positive fraction **falling** in sepsis for SEP-2 and SEP-3-NK even
  though the exact-label fraction rises; generated cells fall too. The earlier 'direction reverses' compared different quantities.
* For the four usable claims with a generated parent (SEP-1, SEP-2, SEP-3-NK, SEP-4B) generated contrasts have the same sign as real;
  threshold-free score contrasts have the same sign for all seven.
* Levels do not match: generated marker scores sit lower than real ones, so classifier-positive fractions are 1.6–9x lower.
* Only 7 adult healthy-control samples exist in the real data, so real contrasts have wide intervals.